In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
np.random.seed(42)
n_samples = 10000

In [4]:
amount = np.random.exponential(scale=100, size=n_samples) + 1
time_of_day = np.random.randint(0, 24, size=n_samples)
is_international = np.random.binomial(1, 0.05, size=n_samples)

fraud_prob = 1 / (1 + np.exp(-( -5 + 0.005 * amount + 0.2 * time_of_day + 2 * is_international )))
y = np.random.binomial(1, fraud_prob)

df = pd.DataFrame({'amount': amount, 'time': time_of_day, 'international': is_international, 'is_fraud': y})
print(f"Total transactions: {len(df)}. Fraudulent: {y.sum()} ({y.mean()*100:.2f}%)")

X_train, X_test, y_train, y_test = train_test_split(df.drop('is_fraud', axis=1), y, test_size=0.3, random_state=42)

COST_FN = 500
COST_FP = 20

Total transactions: 10000. Fraudulent: 1819 (18.19%)


In [6]:
def calculate_custom_loss(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = (fn * COST_FN) + (fp * COST_FP)
    return total_cost, fn, fp

random_preds = np.random.binomial(1, 0.05, size=len(y_test))
cost_rand, fn_rand, fp_rand = calculate_custom_loss(y_test, random_preds)
print(f"\n--- Random Algorithm (blocks 5% randomly) ---")
print(f"Missed fraud (FN): {fn_rand}, Blocked legitimate (FP): {fp_rand}")
print(f"TOTAL LOSS: ${cost_rand}")


--- Random Algorithm (blocks 5% randomly) ---
Missed fraud (FN): 529, Blocked legitimate (FP): 121
TOTAL LOSS: $266920


In [7]:
rule_preds = ((X_test['amount'] > 300) & (X_test['time'] < 5)).astype(int)
cost_rule, fn_rule, fp_rule = calculate_custom_loss(y_test, rule_preds)
print(f"\n--- Primitive Rule-based Algorithm (amount > 300 and time < 5) ---")
print(f"Missed fraud (FN): {fn_rule}, Blocked legitimate (FP): {fp_rule}")
print(f"TOTAL LOSS: ${cost_rule}")


--- Primitive Rule-based Algorithm (amount > 300 and time < 5) ---
Missed fraud (FN): 544, Blocked legitimate (FP): 27
TOTAL LOSS: $272540


In [8]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

best_cost = float('inf')
best_threshold = 0
best_fn, best_fp = 0, 0

for thresh in np.arange(0.1, 0.9, 0.05):
    preds = (probs >= thresh).astype(int)
    cost, fn, fp = calculate_custom_loss(y_test, preds)
    if cost < best_cost:
        best_cost = cost
        best_threshold = thresh
        best_fn, best_fp = fn, fp

print(f"\n--- ML Model (with threshold optimization) ---")
print(f"Optimal probability threshold: {best_threshold:.2f}")
print(f"Missed fraud (FN): {best_fn}, Blocked legitimate (FP): {best_fp}")
print(f"TOTAL LOSS: ${best_cost}")

print("\nCONCLUSION:")
print(f"Compared to random triggering, we saved ${cost_rand - best_cost}")


--- ML Model (with threshold optimization) ---
Optimal probability threshold: 0.10
Missed fraud (FN): 47, Blocked legitimate (FP): 1224
TOTAL LOSS: $47980

CONCLUSION:
Compared to random triggering, we saved $218940
